# Axe 2 : le sentiment de stagnation — exploration

**Question pour la DRH :** qui se sent stagner, à quel moment de la carrière, et pourquoi ?

Ce notebook sert à explorer les données avant d'écrire `analyse/axe_2.py`.
Étape 1 : prendre en main la colonne `JobSatComplacent`, ses réponses et le nombre de personnes qui y ont répondu.

## 1. Charger la population commune

In [1]:
import sys

# le notebook est dans notebooks/ : on ajoute la racine du projet pour importer analyse/
sys.path.append("..")

import pandas as pd
from analyse.chargement import charger
from analyse.population import preparer

df = preparer(charger(2026))
df.shape

(22839, 216)

Zone par défaut du projet : France et pays frontaliers ensemble (voir `ORGANISATION.md`).
La France seule a trop peu de répondants dès qu'on ajoute un filtre.

In [2]:
ZONE_DEFAUT = ["France", "Pays frontaliers"]

zone = df[df["Zone"].isin(ZONE_DEFAUT)]
print("Personnes en France et pays frontaliers :", len(zone))

Personnes en France et pays frontaliers : 3376


## 2. La colonne `JobSatComplacent` et ses réponses

In [3]:
zone["JobSatComplacent"].value_counts(dropna=False)

JobSatComplacent
NaN                                                        1768
I do not feel complacent                                    624
Burnout/tech fatigue                                        292
Economic uncertainty making me hesitant to switch roles     262
AI automating the challenging parts of my job               172
Lack of upward mobility due to corporate tightening         138
Other, please specify                                       120
Name: count, dtype: int64

In [4]:
# une seule réponse par personne, ou plusieurs séparées par « ; » ?
print("Réponses multiples :", zone["JobSatComplacent"].dropna().str.contains(";").sum())

Réponses multiples : 0


In [5]:
# traduction pour l'affichage (consignes/saad_axe-2.md)
TRADUCTION = {
    "I do not feel complacent": "Pas de sentiment de stagnation",
    "Burnout/tech fatigue": "Épuisement",
    "Economic uncertainty making me hesitant to switch roles": "Incertitude économique",
    "AI automating the challenging parts of my job": "L'IA fait les tâches intéressantes",
    "Lack of upward mobility due to corporate tightening": "Pas de perspective d'évolution",
    "Other, please specify": "Autre",
}

reponses = zone["JobSatComplacent"].dropna().map(TRADUCTION)
tableau = pd.DataFrame({
    "repondants": reponses.value_counts(),
    "part": reponses.value_counts(normalize=True).round(3),
})
tableau

,repondants,part
JobSatComplacent,,
Pas de sentiment de stagnation,624,0.388
Épuisement,292,0.182
Incertitude économique,262,0.163
L'IA fait les tâches intéressantes,172,0.107
Pas de perspective d'évolution,138,0.086
Autre,120,0.075


## 3. Qui a répondu à la question ?

In [6]:
def taux_reponse(tableau):
    # nombre de personnes, nombre de réponses à la question, et part de réponses
    return pd.Series({
        "personnes": len(tableau),
        "repondants": tableau["JobSatComplacent"].notna().sum(),
        "taux": round(tableau["JobSatComplacent"].notna().mean(), 3),
    })

pd.DataFrame({
    "France et pays frontaliers": taux_reponse(zone),
    "France seule": taux_reponse(df[df["Zone"] == "France"]),
    "Toute la population": taux_reponse(df),
}).T.astype({"personnes": int, "repondants": int})

,personnes,repondants,taux
France et pays frontaliers,3376,1608,0.476
France seule,612,284,0.464
Toute la population,22839,7540,0.330


In [7]:
# le taux de réponse change-t-il avec l'expérience ?
(zone.groupby("Tranche_experience", observed=True)
     .apply(taux_reponse)
     .astype({"personnes": int, "repondants": int}))

,personnes,repondants,taux
Tranche_experience,,,
0-1,92,51,0.554
2-3,249,119,0.478
4-6,446,227,0.509
7-10,617,295,0.478
11-20,1111,536,0.482
21+,832,377,0.453


In [8]:
# ceux qui ont répondu ont-ils aussi donné leur satisfaction (JobSat) ?
pd.crosstab(zone["JobSat"].notna(), zone["JobSatComplacent"].notna(),
            rownames=["JobSat renseigné"], colnames=["A répondu à JobSatComplacent"])

A répondu à JobSatComplacent,False,True
JobSat renseigné,,
False,20,0
True,1748,1608


## 4. Je remarque / je me demande

**Je remarque**

- Seules **1 608 personnes sur 3 376** ont répondu en France et pays frontaliers, soit **47,6 %**. Sur toute la population, c'est **33,0 %**. Les consignes parlent de « 45 % » : c'est un arrondi, les vrais chiffres sont ceux-là.
- **Une seule réponse par personne** : pas de « ; ». Les parts des réponses font donc 100 % à elles toutes.
- Parmi ceux qui ont répondu, **38,8 %** ne se sentent pas en stagnation. Donc **61,2 %** citent une cause.
- La première cause est **l'épuisement (18,2 %)**, puis **l'incertitude économique (16,3 %)**. Le manque de perspective d'évolution n'arrive qu'en 5ᵉ position (8,6 %), alors que le message de départ du projet parle surtout d'évolution : à vérifier dans la suite.
- Le taux de réponse varie peu avec l'expérience, entre 45 % et 55 % : pas de tranche qui répond beaucoup plus que les autres.
- La tranche **0-1 an** n'a que 92 personnes et **51 répondants**, juste au-dessus du seuil de 50. Avec un filtre de secteur, elle passera sous le seuil.
- En **France seule**, il n'y a que **284 répondants** : la zone par défaut France et pays frontaliers est nécessaire pour cet axe.
- Tous ceux qui ont répondu ont aussi donné leur satisfaction (`JobSat`) : on pourra croiser les deux.

**Je me demande**

- Pourquoi plus de la moitié n'ont pas répondu ? Si ceux qui se sentent bloqués ont plus répondu, la part de 61,2 % est trop haute. Il faut l'écrire sur la page.
- La part en stagnation monte-t-elle avec l'expérience, et à partir de quelle tranche ? (prochaine étape)
- Les causes sont-elles différentes en ESN et chez les éditeurs de logiciels ? (prochaine étape)
- « Complacent » veut plutôt dire « s'installer, ne plus chercher à bouger ». Le mot « stagnation » choisi par le groupe est-il le bon ? À discuter avec le groupe.

## 5. Qui se sent stagner, et pourquoi ?

On ne garde que les personnes qui ont répondu à la question.
Une personne est **en stagnation** si elle a donné une cause, c'est-à-dire toute réponse sauf « Pas de sentiment de stagnation ».

In [9]:
import plotly.express as px
import plotly.io as pio

# les figures chargent plotly depuis internet : le notebook reste léger
pio.renderers.default = "notebook_connected"

# couleurs provisoires, en attendant GRIS et ACCENT de analyse/affichage.py (Gaetan)
GRIS = "#B0B0B0"
ACCENT = "#C0392B"

repondu = zone[zone["JobSatComplacent"].notna()].copy()
repondu["Reponse"] = repondu["JobSatComplacent"].map(TRADUCTION)
repondu["En_stagnation"] = repondu["Reponse"] != "Pas de sentiment de stagnation"

print("Répondants :", len(repondu), "· en stagnation :", repondu["En_stagnation"].sum(),
      f"({repondu['En_stagnation'].mean():.1%})")

Répondants : 1608 · en stagnation : 984 (61.2%)


### 5.1 La part en stagnation selon l'expérience

In [10]:
par_experience = (repondu.groupby("Tranche_experience", observed=True)["En_stagnation"]
                  .agg(part="mean", repondants="size")
                  .reset_index())
par_experience["part"] = par_experience["part"].round(3)
par_experience

,Tranche_experience,part,repondants
0,0-1,0.471,51
1,2-3,0.639,119
2,4-6,0.652,227
3,7-10,0.678,295
4,11-20,0.619,536
5,21+,0.541,377


In [11]:
fig = px.bar(par_experience, x="Tranche_experience", y="part", text_auto=".0%",
             labels={"Tranche_experience": "Années d'expérience", "part": "Part en stagnation"},
             title="Dès 2 ans d'expérience, près de deux répondants sur trois se sentent stagner")
# la tranche la plus haute en ACCENT, le reste en gris
couleurs = [ACCENT if p == par_experience["part"].max() else GRIS for p in par_experience["part"]]
fig.update_traces(marker_color=couleurs)
fig.update_yaxes(tickformat=".0%")
fig.show()
print(f"Développeurs en emploi en France et pays frontaliers · 2026 · {len(repondu)} répondants à la question")

Développeurs en emploi en France et pays frontaliers · 2026 · 1608 répondants à la question


### 5.2 Les causes, parmi les personnes en stagnation

In [12]:
en_stagnation = repondu[repondu["En_stagnation"]]

causes = (en_stagnation["Reponse"].value_counts(normalize=True).round(3)
          .rename("part").rename_axis("Cause").reset_index())
causes["repondants"] = causes["Cause"].map(en_stagnation["Reponse"].value_counts())
causes

,Cause,part,repondants
0,Épuisement,0.297,292
1,Incertitude économique,0.266,262
2,L'IA fait les tâches intéressantes,0.175,172
3,Pas de perspective d'évolution,0.140,138
4,Autre,0.122,120


In [13]:
fig = px.bar(causes.sort_values("part"), x="part", y="Cause", orientation="h", text_auto=".0%",
             labels={"part": "Part des personnes en stagnation", "Cause": ""},
             title="L'épuisement est la première cause, devant l'incertitude économique")
fig.update_traces(marker_color=[ACCENT if c == causes["Cause"].iloc[0] else GRIS
                                for c in causes.sort_values("part")["Cause"]])
fig.update_xaxes(tickformat=".0%")
fig.show()
print(f"Personnes en stagnation en France et pays frontaliers · 2026 · {len(en_stagnation)} répondants")

Personnes en stagnation en France et pays frontaliers · 2026 · 984 répondants


### 5.3 Les causes selon le secteur

In [14]:
# part en stagnation par secteur, avec le nombre de répondants
(repondu.groupby("Secteur")["En_stagnation"]
        .agg(part="mean", repondants="size")
        .round(3))

,part,repondants
Secteur,,
Autre secteur,0.602,1015
ESN,0.646,294
Éditeur de logiciels,0.612,299


In [15]:
# les causes (en lignes) selon le secteur (en colonnes), parmi les personnes en stagnation
causes_secteur = pd.crosstab(en_stagnation["Reponse"], en_stagnation["Secteur"], normalize="columns").round(3)
print("Personnes en stagnation par secteur :", en_stagnation["Secteur"].value_counts().to_dict())
causes_secteur

Personnes en stagnation par secteur : {'Autre secteur': 611, 'ESN': 190, 'Éditeur de logiciels': 183}


Secteur,Autre secteur,ESN,Éditeur de logiciels
Reponse,,,
Autre,0.131,0.111,0.104
Incertitude économique,0.265,0.289,0.246
L'IA fait les tâches intéressantes,0.167,0.147,0.230
Pas de perspective d'évolution,0.141,0.147,0.131
Épuisement,0.296,0.305,0.290


In [16]:
a_tracer = causes_secteur.reset_index().melt(id_vars="Reponse", var_name="Secteur", value_name="part")
fig = px.bar(a_tracer, x="part", y="Reponse", color="Secteur", barmode="group", orientation="h",
             text_auto=".0%", labels={"part": "Part des personnes en stagnation", "Reponse": ""},
             color_discrete_map={"ESN": ACCENT, "Éditeur de logiciels": "#7F7F7F", "Autre secteur": GRIS},
             title="Les causes se ressemblent d'un secteur à l'autre ; l'IA pèse plus chez les éditeurs")
fig.update_xaxes(tickformat=".0%")
fig.show()
print(f"Personnes en stagnation en France et pays frontaliers · 2026 · {len(en_stagnation)} répondants")

Personnes en stagnation en France et pays frontaliers · 2026 · 984 répondants


### 5.4 Deux contrôles

In [17]:
# les causes selon l'expérience : combien de personnes en stagnation par tranche ?
en_stagnation.groupby("Tranche_experience", observed=True).size().rename("en_stagnation")

Tranche_experience
0-1       24
2-3       76
4-6      148
7-10     200
11-20    332
21+      204
Name: en_stagnation, dtype: int64

In [18]:
# la satisfaction (JobSat, de 0 à 10) est-elle plus basse chez ceux qui se sentent stagner ?
repondu.groupby("En_stagnation")["JobSat"].agg(["mean", "size"]).round(2)

,mean,size
En_stagnation,,
False,7.63,624
True,7.48,984


## 6. Je remarque / je me demande (étape 2)

**Je remarque**

- **47,1 %** de stagnation à 0-1 an (51 répondants), puis **63,9 %** dès 2-3 ans. La part reste haute jusqu'à **67,8 %** à 7-10 ans, puis redescend à **54,1 %** après 21 ans. Le palier se fait bien **vers 2 ans**, c'est le moment où l'ESN perd ses développeurs.
- Parmi les **984 personnes en stagnation**, les causes sont : **épuisement 29,7 %**, **incertitude économique 26,6 %**, IA 17,5 %, pas de perspective d'évolution 14,0 %, autre 12,2 %.
- L'**évolution n'est que la 4ᵉ cause**. Le message de départ (« il faut leur offrir de quoi évoluer ») n'est pas confirmé par cet axe : l'épuisement, donc la charge de travail, ressort davantage.
- Les secteurs sont proches : **64,6 %** de stagnation en ESN (294 répondants), **61,2 %** chez les éditeurs (299), **60,2 %** ailleurs (1 015). L'écart ESN / éditeurs est faible.
- En ESN, les causes ressemblent aux autres secteurs : épuisement 30,5 %, incertitude 28,9 %. Chez les éditeurs, l'IA pèse plus : **23,0 %**, contre 14,7 % en ESN.
- Les causes par tranche reposent sur de petits effectifs : seulement **24 personnes en stagnation à 0-1 an** et 76 à 2-3 ans. À 0-1 an, on ne pourra pas afficher les causes (seuil de 50).
- La satisfaction moyenne est à peine plus basse chez ceux qui se sentent stagner : **7,48** contre **7,63** sur 10.

**Je me demande**

- Faut-il dire au groupe que l'évolution n'est pas la première cause ? Ça touche le message du tableau de bord et l'axe 4 de Gaetan.
- Pour la vue 3, faut-il afficher les causes par secteur, alors que les écarts sont petits ? Ou plutôt « ESN contre le reste » ?
- Le filtre de tranche d'expérience de la page devra masquer les causes quand il reste moins de 50 personnes en stagnation.

## 7. Tester les fonctions de `analyse/axe_2.py`

On importe la fonction de l'application et on vérifie qu'elle redonne les chiffres de la section 5.1.

In [19]:
from analyse.axe_2 import part_stagnation_par_experience

resultat = part_stagnation_par_experience(zone)
resultat.round(3)

,Tranche_experience,part,repondants
0,0-1,0.471,51
1,2-3,0.639,119
2,4-6,0.652,227
3,7-10,0.678,295
4,11-20,0.619,536
5,21+,0.541,377


In [20]:
# mêmes chiffres que l'exploration de la section 5.1 ?
pd.testing.assert_frame_equal(
    resultat[resultat["repondants"] > 0].reset_index(drop=True)
            .assign(part=lambda t: t["part"].round(3)),
    par_experience,
    check_dtype=False, check_categorical=False,
)
print("OK : la fonction redonne les chiffres de l'exploration")

OK : la fonction redonne les chiffres de l'exploration


In [21]:
# avec un filtre de secteur : beaucoup de tranches passent sous 50 répondants
part_stagnation_par_experience(zone[zone["Secteur"] == "ESN"]).round(3)

,Tranche_experience,part,repondants
0,0-1,0.500,8
1,2-3,0.643,14
2,4-6,0.795,39
3,7-10,0.723,47
4,11-20,0.649,111
5,21+,0.548,73


**Je remarque :** en ESN seule, aucune tranche n'atteint 50 répondants avant 11-20 ans (111) et 21+ (73). La page devra cacher les chiffres des tranches sous le seuil avec `assez_de_reponses`.